# B7 - Simple Vision Transformer (ViT) on MNIST
Patches 7x7 (16 x 49) -> Dense64 + positional embedding -> MHA (4 heads) + residual -> LayerNorm -> FFN (128 GELU, 64) + residual -> GAP -> Dense64 ReLU -> Dense10 softmax

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
# Load MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") / 255.0)[..., None]
x_test = (x_test.astype("float32") / 255.0)[..., None]
PATCH, NUM_PATCHES, DIM = 7, 16, 64

In [ ]:
class Patches(layers.Layer):
    """Split image into 7x7 patches and flatten -> (16, 49)"""
    def __init__(self, patch_size, **kw):
        super().__init__(**kw)
        self.p = patch_size

    def call(self, images):
        patches = tf.image.extract_patches(images, sizes=[1, self.p, self.p, 1], strides=[1, self.p, self.p, 1],
                                           rates=[1, 1, 1, 1], padding='VALID')
        return tf.reshape(patches, [-1, NUM_PATCHES, self.p * self.p])


class PatchEncoder(layers.Layer):
    """Dense(64) patch embedding + learned positional embedding"""
    def __init__(self, num_patches, dim, **kw):
        super().__init__(**kw)
        self.num_patches = num_patches
        self.proj = layers.Dense(dim)
        self.pos_emb = layers.Embedding(num_patches, dim)

    def call(self, patches):
        positions = tf.range(self.num_patches)
        return self.proj(patches) + self.pos_emb(positions)

In [ ]:
def build_vit():
    inp = layers.Input((28, 28, 1))
    x = Patches(PATCH, name='patch_extraction')(inp)                 # 16 x 49
    x = PatchEncoder(NUM_PATCHES, DIM, name='patch_pos_embedding')(x) # 16 x 64

    attn = layers.MultiHeadAttention(num_heads=4, key_dim=DIM // 4)(x, x)  # 16 x 64
    x = layers.Add()([x, attn])                                       # residual
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(128, activation='gelu')(x)                     # 16 x 128
    ffn = layers.Dense(64)(ffn)                                       # 16 x 64
    x = layers.Add()([x, ffn])                                        # residual

    x = layers.GlobalAveragePooling1D()(x)                            # 64
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(10, activation='softmax')(x)
    return keras.Model(inp, out, name='vit')

model = build_vit()
model.summary()

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1)

In [ ]:
# Training loss vs epochs
plt.plot(history.history['loss'], label='train'); plt.plot(history.history['val_loss'], label='val')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('ViT training loss'); plt.show()

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

# One test image with actual and predicted digit
i = 0
pred = np.argmax(model.predict(x_test[i:i+1], verbose=0))
plt.imshow(x_test[i, :, :, 0], cmap='gray'); plt.title(f"Actual: {y_test[i]}   Predicted: {pred}"); plt.axis('off'); plt.show()